### Historical experiment notebook

Saved outputs reflect earlier execution sessions and dataset availability.
Paths and label-length handling were updated during repository cleanup.
The notebook has not been rerun after these changes.

In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "requirements-asr.txt").is_file():
        requirements_file = candidate / "requirements-asr.txt"
        break
else:
    raise FileNotFoundError("Could not find requirements-asr.txt")

%pip install -r "{requirements_file}"
# =====


In [ ]:
# =====
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (
        (candidate / "1_ASR").is_dir()
        and (candidate / "2_QA_GENERATION").is_dir()
    ):
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Open the notebook from inside the project folder."
    )

ASR_DIR = PROJECT_ROOT / "1_ASR"
DATASET_DIR = PROJECT_ROOT / "NileTTS-dataset"
DATA_ROOT = str(DATASET_DIR)
# =====


In [ ]:

from huggingface_hub import snapshot_download
import pandas as pd
from pathlib import Path
import os
import re

repo_id = "KickItLikeShika/NileTTS-dataset"

snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    # =====
    # local_dir="NileTTS-dataset",
    local_dir=str(DATASET_DIR),
    # =====
    allow_patterns=[
        "metadata_train.csv",
        "metadata_eval.csv",
        "wavs/*.wav"
    ]
)

# =====
# root = Path("NileTTS-dataset")
root = DATASET_DIR
# =====

train_df = pd.read_csv(root / "metadata_train.csv", sep="|")
eval_df  = pd.read_csv(root / "metadata_eval.csv",  sep="|")

train_df["audio_file"] = train_df["audio_file"].str.replace("wav/", "wavs/", regex=False)
eval_df["audio_file"]  = eval_df["audio_file"].str.replace("wav/",  "wavs/", regex=False)

train_df["audio_path"] = train_df["audio_file"].apply(lambda x: str(root / x))
eval_df["audio_path"]  = eval_df["audio_file"].apply(lambda x: str(root / x))

print("Train shape:", train_df.shape)
print("Eval shape:",  eval_df.shape)
print("First audio exists:", os.path.exists(train_df.loc[0, "audio_path"]))

# =====
# train_df.to_csv("niletts_train_clean.csv", index=False)
# eval_df.to_csv("niletts_eval_clean.csv",   index=False)

# The dataframes are used directly by the following cells.
# =====

train_df.head()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you h

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

HTTP Error 504 thrown while requesting HEAD https://huggingface.co/datasets/KickItLikeShika/NileTTS-dataset/resolve/6905cb892bda2a315377178729ab2e718c0e63d4/wavs/general_22_00037.wav
Retrying in 1s [Retry 1/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/datasets/KickItLikeShika/NileTTS-dataset/resolve/6905cb892bda2a315377178729ab2e718c0e63d4/wavs/general_22_00037.wav
Retrying in 2s [Retry 2/5].


Train shape: (8571, 4)
Eval shape: (950, 4)
First audio exists: True


,audio_file,text,speaker_name,audio_path
0,wavs/general_1_00041.wav,تماما وبجانب درجة الجودة العامة بيسألهم أسئلة ...,SPEAKER_02,NileTTS-dataset/wavs/general_1_00041.wav
1,wavs/medical_8_00009.wav,لأن العواقب أسوأ بكتير من اللي بنتخيله بص، الل...,SPEAKER_02,NileTTS-dataset/wavs/medical_8_00009.wav
2,wavs/sales_15_00036.wav,بنبص بره على العميل و القيمة اللي بيحصل عليها....,SPEAKER_01,NileTTS-dataset/wavs/sales_15_00036.wav
3,wavs/sales_3_00038.wav,تاني حاجة خططك كلها بتتدمر. حلم إنك تستقر في ب...,SPEAKER_02,NileTTS-dataset/wavs/sales_3_00038.wav
4,wavs/sales_26_00088.wav,هنا الأمور بدأت تتغير بشكل ملموس قوي، الإنترنت...,SPEAKER_02,NileTTS-dataset/wavs/sales_26_00088.wav


In [3]:
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

print("Whisper imports are working.")

Whisper imports are working.


In [ ]:
# =====
# import shutil
# import os

# DATA_ROOT = "/content/NileTTS-dataset"  # change if needed
# BACKUP_PATH = "/content/drive/MyDrive/NileTTS-dataset"

# if os.path.exists(DATA_ROOT):
#     if not os.path.exists(BACKUP_PATH):
#         shutil.copytree(DATA_ROOT, BACKUP_PATH)
#         print("Copied dataset to Drive.")
#     else:
#         print("Backup already exists.")
# else:
#     print("Dataset folder not found. Check DATA_ROOT.")
# =====


Copied dataset to Drive.


In [ ]:
# =====
# from google.colab import drive
# drive.mount("/content/drive")
# =====


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import numpy as np
import pandas as pd
import pyarrow as pa
import datasets
import transformers
import sklearn
import scipy
import accelerate

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("sklearn:", sklearn.__version__)
print("scipy:", scipy.__version__)
print("accelerate:", accelerate.__version__)

numpy: 1.26.4
pandas: 2.2.2
pyarrow: 15.0.2
datasets: 2.19.2
transformers: 4.41.2
sklearn: 1.4.2
scipy: 1.13.1
accelerate: 1.2.1


In [12]:
from datasets import Dataset, DatasetDict, Audio

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

print("All imports are working.")

All imports are working.


In [ ]:
# =====
# import os

# DATA_ROOT_DRIVE = "/content/drive/MyDrive/Datasets/NileTTS-dataset"

# print(os.path.exists(DATA_ROOT_DRIVE))
# print(os.listdir(DATA_ROOT_DRIVE)[:20])
# =====


True
['wavs', 'metadata_train.csv', 'metadata_eval.csv', '.cache']


In [ ]:
# =====
# import os

# print("DATA_ROOT_DRIVE =", DATA_ROOT_DRIVE)
# print("Exists:", os.path.exists(DATA_ROOT_DRIVE))

# if os.path.exists(DATA_ROOT_DRIVE):
#     print("Contents:")
#     print(os.listdir(DATA_ROOT_DRIVE)[:50])
# =====


DATA_ROOT_DRIVE = /content/drive/MyDrive/Datasets/NileTTS-dataset
Exists: True
Contents:
['wavs', 'metadata_train.csv', 'metadata_eval.csv', '.cache']


In [ ]:
# =====
# import os
# import pandas as pd

# DATA_ROOT_DRIVE = "/content/drive/MyDrive/Datasets/NileTTS-dataset"

# train_csv = os.path.join(DATA_ROOT_DRIVE, "metadata_train.csv")
# eval_csv = os.path.join(DATA_ROOT_DRIVE, "metadata_eval.csv")

# print("Train CSV exists:", os.path.exists(train_csv))
# print("Eval CSV exists:", os.path.exists(eval_csv))
# print("Dataset contents:", os.listdir(DATA_ROOT_DRIVE)[:20])

# train_df = pd.read_csv(train_csv, sep="|")
# eval_df = pd.read_csv(eval_csv, sep="|")

# print("Train rows:", len(train_df))
# print("Eval rows:", len(eval_df))
# print("Columns:", train_df.columns.tolist())

# train_df.head()
# =====


Train CSV exists: True
Eval CSV exists: True
Dataset contents: ['wavs', 'metadata_train.csv', 'metadata_eval.csv', '.cache']
Train rows: 8571
Eval rows: 950
Columns: ['audio_file', 'text', 'speaker_name']


,audio_file,text,speaker_name
0,wavs/general_1_00041.wav,تماما وبجانب درجة الجودة العامة بيسألهم أسئلة ...,SPEAKER_02
1,wavs/medical_8_00009.wav,لأن العواقب أسوأ بكتير من اللي بنتخيله بص، الل...,SPEAKER_02
2,wavs/sales_15_00036.wav,بنبص بره على العميل و القيمة اللي بيحصل عليها....,SPEAKER_01
3,wavs/sales_3_00038.wav,تاني حاجة خططك كلها بتتدمر. حلم إنك تستقر في ب...,SPEAKER_02
4,wavs/sales_26_00088.wav,هنا الأمور بدأت تتغير بشكل ملموس قوي، الإنترنت...,SPEAKER_02


In [ ]:
# =====
# import shutil
# import os

# DATA_ROOT_DRIVE = "/content/drive/MyDrive/Datasets/NileTTS-dataset"
# DATA_ROOT = "/content/NileTTS-dataset"

# if os.path.exists(DATA_ROOT):
#     shutil.rmtree(DATA_ROOT)

# shutil.copytree(DATA_ROOT_DRIVE, DATA_ROOT)

# print("Copied to /content")
# print(os.listdir(DATA_ROOT)[:20])
# =====


KeyboardInterrupt: 

In [ ]:
# =====
# import os

# DATA_ROOT_DRIVE = "/content/drive/MyDrive/Datasets/NileTTS-dataset"
# DATA_ROOT = "/content/NileTTS-dataset"

# if os.path.exists(DATA_ROOT):
#     print("Using existing /content copy:", DATA_ROOT)
# else:
#     print("No /content copy found. Use Drive path directly for now.")

# print("Drive path exists:", os.path.exists(DATA_ROOT_DRIVE))
# print("Content path exists:", os.path.exists(DATA_ROOT))

print("Dataset directory:", DATASET_DIR)
print("Dataset directory exists:", DATASET_DIR.is_dir())
# =====

Using existing /content copy: /content/NileTTS-dataset
Drive path exists: True
Content path exists: True


In [ ]:
import os

# =====
# DATA_ROOT = "/content/drive/MyDrive/Datasets/NileTTS-dataset"
DATA_ROOT = str(DATASET_DIR)
# =====

def fix_audio_path(path):
    path = str(path).strip().replace("\\", "/")

    # Your metadata already has paths like wavs/general_1_00041.wav
    if path.startswith("wav/"):
        path = path.replace("wav/", "wavs/", 1)

    if "/" not in path:
        path = f"wavs/{path}"

    return os.path.join(DATA_ROOT, path)


train_df["audio"] = train_df["audio_file"].apply(fix_audio_path)
eval_df["audio"] = eval_df["audio_file"].apply(fix_audio_path)

train_df["exists"] = train_df["audio"].apply(os.path.exists)
eval_df["exists"] = eval_df["audio"].apply(os.path.exists)

print("Train existing audio:", train_df["exists"].sum(), "/", len(train_df))
print("Eval existing audio:", eval_df["exists"].sum(), "/", len(eval_df))
print("Total existing:", train_df["exists"].sum() + eval_df["exists"].sum())

print(train_df[["audio_file", "audio", "exists"]].head())

Train existing audio: 5383 / 8571
Eval existing audio: 565 / 950
Total existing: 5948
                 audio_file  \
0  wavs/general_1_00041.wav   
1  wavs/medical_8_00009.wav   
2   wavs/sales_15_00036.wav   
3    wavs/sales_3_00038.wav   
4   wavs/sales_26_00088.wav   

                                               audio  exists  
0  /content/drive/MyDrive/Datasets/NileTTS-datase...   False  
1  /content/drive/MyDrive/Datasets/NileTTS-datase...   False  
2  /content/drive/MyDrive/Datasets/NileTTS-datase...   False  
3  /content/drive/MyDrive/Datasets/NileTTS-datase...   False  
4  /content/drive/MyDrive/Datasets/NileTTS-datase...    True  


In [ ]:
# =====
missing_train = int((~train_df["exists"]).sum())
missing_eval = int((~eval_df["exists"]).sum())

if missing_train or missing_eval:
    raise FileNotFoundError(
        f"Dataset audio is incomplete: "
        f"{missing_train} missing train files and "
        f"{missing_eval} missing eval files. "
        f"Check DATA_ROOT and complete the dataset download."
    )
# =====

train_df = train_df[train_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)
eval_df = eval_df[eval_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)

print("Final train size:", len(train_df))
print("Final eval size:", len(eval_df))
print("Columns:", train_df.columns.tolist())

Final train size: 5383
Final eval size: 565
Columns: ['audio_file', 'text', 'speaker_name', 'audio']


In [8]:
from datasets import Dataset, DatasetDict, Audio

print("Dataset imports are working.")

Dataset imports are working.


In [9]:
import re
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


train_df["text"] = train_df["text"].apply(clean_text)
eval_df["text"] = eval_df["text"].apply(clean_text)

train_df = train_df[train_df["text"].str.len() > 0].reset_index(drop=True)
eval_df = eval_df[eval_df["text"].str.len() > 0].reset_index(drop=True)

train_df[["audio_file", "text", "speaker_name"]].head()

,audio_file,text,speaker_name
0,wavs/sales_26_00088.wav,هنا الأمور بدأت تتغير بشكل ملموس قوي، الإنترنت...,SPEAKER_02
1,wavs/sales_5_00170.wav,بالزبط الأساس: التلاجة، الغسالة، البوتاجاز، ال...,SPEAKER_02
2,wavs/medical_5_00068.wav,سؤال يستحق أننا نفكر فيه,SPEAKER_01
3,wavs/sales_33_00043.wav,طيب فيه فكرة كمان لفتت نظري في المصادر. و ممكن...,SPEAKER_02
4,wavs/general_44_00031.wav,آه الجملة دي الوحدها تخلي الواحد يعترف بجرائم ...,SPEAKER_02


In [10]:
from datasets import Dataset, DatasetDict, Audio

train_dataset = Dataset.from_pandas(
    train_df[["audio", "text", "audio_file", "speaker_name"]],
    preserve_index=False
)

eval_dataset = Dataset.from_pandas(
    eval_df[["audio", "text", "audio_file", "speaker_name"]],
    preserve_index=False
)

dataset = DatasetDict({
    "train": train_dataset,
    "eval": eval_dataset
})

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'audio_file', 'speaker_name'],
        num_rows: 5383
    })
    eval: Dataset({
        features: ['audio', 'text', 'audio_file', 'speaker_name'],
        num_rows: 565
    })
})

In [13]:
MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="Arabic",
    task="transcribe"
)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.generation_config.language = "arabic"
model.generation_config.task = "transcribe"

model.config.use_cache = False

print("Model loaded:", MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Model loaded: openai/whisper-small


In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    # =====
    # batch["labels"] = processor.tokenizeoutput_dir="./whisper-small-niletts-egyptian-arabic",r(batch["text"]).input_ids
    batch["labels"] = processor.tokenizer(
        batch["text"],
        truncation=True,
        max_length=448
    ).input_ids
    # =====

    return batch

In [20]:
small_dataset = DatasetDict({
    "train": dataset["train"].select(range(min(1000, len(dataset["train"])))),
    "eval": dataset["eval"].select(range(min(200, len(dataset["eval"]))))
})
processed_train = small_dataset["train"].map(
    prepare_dataset,
    remove_columns=small_dataset["train"].column_names,
    num_proc=1,
    writer_batch_size=8,
    keep_in_memory=False,
    load_from_cache_file=False,
    desc="Processing small train set"
)

processed_eval = small_dataset["eval"].map(
    prepare_dataset,
    remove_columns=small_dataset["eval"].column_names,
    num_proc=1,
    writer_batch_size=8,
    keep_in_memory=False,
    load_from_cache_file=False,
    desc="Processing small eval set"
)

processed_dataset = DatasetDict({
    "train": processed_train,
    "eval": processed_eval
})

Processing small train set:   0%|          | 0/1000 [00:00<?, ? examples/s]

Processing small eval set:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
processed_train = dataset["train"].map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1,
    writer_batch_size=16,
    keep_in_memory=False,
    load_from_cache_file=False,
    desc="Processing train set"
)

processed_eval = dataset["eval"].map(
    prepare_dataset,
    remove_columns=dataset["eval"].column_names,
    num_proc=1,
    writer_batch_size=16,
    keep_in_memory=False,
    load_from_cache_file=False,
    desc="Processing eval set"
)

processed_dataset = DatasetDict({
    "train": processed_train,
    "eval": processed_eval
})

processed_dataset

Processing train set:   0%|          | 0/5383 [00:00<?, ? examples/s]

In [1]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        input_features = [
            {"input_features": feature["input_features"]}
            for feature in features
        ]

        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [ ]:
def normalize_arabic_for_eval(text):
    text = str(text)

    # Remove tashkeel
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    # Normalize Arabic letters
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)

    # Remove punctuation but keep Arabic/English letters and numbers
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = processor.tokenizer.batch_decode(
        label_ids,
        skip_special_tokens=True
    )

    pred_str = [normalize_arabic_for_eval(x) for x in pred_str]
    label_str = [normalize_arabic_for_eval(x) for x in label_str]

    wer = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

In [ ]:
training_args = Seq2SeqTrainingArguments(
    # =====
    # output_dir="./whisper-small-niletts-egyptian-arabic",
    output_dir=str(ASR_DIR / "whisper-small-niletts-egyptian-arabic"),
    # =====

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=3,

    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),

    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    logging_steps=25,

    predict_with_generate=True,
    generation_max_length=225,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    report_to="none",
    remove_unused_columns=False
)

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["eval"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    # =====
    # processing_class=processor
    tokenizer=processor.tokenizer
    # =====
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()

eval_results

In [ ]:
# =====
# FINAL_MODEL_DIR = "./whisper-small-niletts-egyptian-arabic-final"
FINAL_MODEL_DIR = str(ASR_DIR / "whisper-small-niletts-egyptian-arabic-final")
# =====

trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)

print("Saved to:", FINAL_MODEL_DIR)

In [ ]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=FINAL_MODEL_DIR,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device
)

sample_audio = eval_df.loc[0, "audio"]
reference_text = eval_df.loc[0, "text"]

prediction = asr_pipe(sample_audio)

print("Reference:")
print(reference_text)

print("\nPrediction:")
print(prediction["text"])

In [ ]:
predictions = []

for i, row in eval_df.iterrows():
    result = asr_pipe(row["audio"])
    pred_text = result["text"]

    predictions.append({
        "audio_file": row["audio_file"],
        "speaker_name": row["speaker_name"],
        "text": row["text"],
        "asr_transcription_our_finetuned_whisper": pred_text
    })

    if (i + 1) % 50 == 0:
        print(f"Done {i + 1}/{len(eval_df)}")

asr_output_df = pd.DataFrame(predictions)

asr_output_df.head()

In [ ]:
def row_wer(ref, hyp):
    ref = normalize_arabic_for_eval(ref)
    hyp = normalize_arabic_for_eval(hyp)
    return wer_metric.compute(predictions=[hyp], references=[ref])


def row_cer(ref, hyp):
    ref = normalize_arabic_for_eval(ref)
    hyp = normalize_arabic_for_eval(hyp)
    return cer_metric.compute(predictions=[hyp], references=[ref])


asr_output_df["wer_our_finetuned_whisper"] = asr_output_df.apply(
    lambda row: row_wer(
        row["text"],
        row["asr_transcription_our_finetuned_whisper"]
    ),
    axis=1
)

asr_output_df["cer_our_finetuned_whisper"] = asr_output_df.apply(
    lambda row: row_cer(
        row["text"],
        row["asr_transcription_our_finetuned_whisper"]
    ),
    axis=1
)

print("Average WER:", asr_output_df["wer_our_finetuned_whisper"].mean())
print("Average CER:", asr_output_df["cer_our_finetuned_whisper"].mean())

asr_output_df.head()

In [ ]:
# =====
# OUTPUT_CSV = "asr_eval_our_finetuned_whisper_small.csv"
OUTPUT_CSV = (ASR_DIR / "asr_eval_our_finetuned_whisper_small.csv")
# =====

asr_output_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_CSV)

In [ ]:
missing_files = eval_df[~eval_df["audio_path"].apply(os.path.exists)]
print(f"Missing files: {len(missing_files)}")
print(missing_files[['audio_path']].head())  # Show the first few missing file paths

Missing files: 0
Empty DataFrame
Columns: [audio_path]
Index: []


In [ ]:
import whisper

def transcribe_audio(model_name, audio_path):
    model = whisper.load_model(model_name)
    result = model.transcribe(audio_path)
    return result['text']

In [ ]:
from tqdm import tqdm

# Ensure tqdm is enabled for pandas
tqdm.pandas()
base_model = "base"
# Apply transcription to eval set with progress bar
eval_df['asr_transcription_base'] = eval_df['audio_path'].progress_apply(
    lambda x: transcribe_audio(base_model, x)
)

# Save the transcriptions
# =====
output_file = ASR_DIR / "asr_eval_transcriptions_with_progress.csv"

# eval_df.to_csv("/content/asr_eval_transcriptions_with_progress.csv", index=False)
eval_df.to_csv(output_file, index=False)

# print("Transcriptions saved to /content/asr_eval_transcriptions_with_progress.csv")
print("Saved:", output_file)
# =====


  0%|          | 0/950 [00:00<?, ?it/s]
  0%|                                               | 0.00/139M [00:00<?, ?iB/s]
 17%|██████▋                                | 23.6M/139M [00:00<00:00, 248MiB/s]
 38%|██████████████▉                        | 52.9M/139M [00:00<00:00, 283MiB/s]
 58%|██████████████████████▋                | 80.5M/139M [00:00<00:00, 286MiB/s]
 78%|███████████████████████████████▏        | 108M/139M [00:00<00:00, 266MiB/s]
100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 227MiB/s]
100%|██████████| 950/950 [41:27<00:00,  2.62s/it]

Transcriptions saved to /content/asr_eval_transcriptions_with_progress.csv


In [ ]:
def normalize_arabic(text):
    text = str(text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)  # remove tashkeel
    text = re.sub(r"ـ", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "ء", text)
    text = re.sub(r"ئ", "ء", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"گ", "ك", text)
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    text = re.sub(r"[0-9٠-٩]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Normalize both reference and ASR transcription
eval_df['ref_norm'] = eval_df['text'].apply(normalize_arabic)
eval_df['asr_norm_base'] = eval_df['asr_transcription_base'].apply(normalize_arabic)

In [ ]:
from jiwer import wer, cer

# Compute WER and CER
eval_df['wer_base'] = eval_df.apply(lambda row: wer(row['ref_norm'], row['asr_norm_base']), axis=1)
eval_df['cer_base'] = eval_df.apply(lambda row: cer(row['ref_norm'], row['asr_norm_base']), axis=1)

# Print average WER and CER for Base model
print(f"Base Model - WER: {eval_df['wer_base'].mean()}")
print(f"Base Model - CER: {eval_df['cer_base'].mean()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.0 MB/s eta 0:00:00
Base Model - WER: 0.5363087785012708
Base Model - CER: 0.17257424279652153


In [ ]:
# Initialize Whisper model (Small)
tqdm.pandas()
small_model = "small"  # Whisper's small model

# Apply transcription to eval set with Small model
eval_df['asr_transcription_small'] = eval_df['audio_path'].progress_apply(lambda x: transcribe_audio(small_model, x))





In [ ]:
# Normalize both reference and ASR transcription for Small model
eval_df['asr_norm_small'] = eval_df['asr_transcription_small'].apply(normalize_arabic)

In [ ]:
# Compute WER and CER for Small model
eval_df['wer_small'] = eval_df.apply(lambda row: wer(row['ref_norm'], row['asr_norm_small']), axis=1)
eval_df['cer_small'] = eval_df.apply(lambda row: cer(row['ref_norm'], row['asr_norm_small']), axis=1)

# Print average WER and CER for Small model
print(f"Small Model - WER: {eval_df['wer_small'].mean()}")
print(f"Small Model - CER: {eval_df['cer_small'].mean()}")

In [ ]:
import torchaudio

# Load a single audio file from your eval dataset
audio_path = eval_df['audio_path'].iloc[0]  # Take the first file
waveform, sample_rate = torchaudio.load(audio_path)

# Check the waveform shape and sample rate
print(f"Waveform Shape: {waveform.shape}")
print(f"Sample Rate: {sample_rate}")

Waveform Shape: torch.Size([2, 440118])
Sample Rate: 44100


In [ ]:
# Resample if necessary (check if sample rate is not 16kHz)
if sample_rate != 16000:
    print(f"Resampling from {sample_rate} Hz to 16kHz")
    waveform = torchaudio.transforms.Resample(sample_rate, 16000)(waveform)
    sample_rate = 16000

# Ensure the waveform is 2D [batch_size, num_frames]
if len(waveform.shape) == 1:
    waveform = waveform.unsqueeze(0)  # Add batch dimension

# Now, waveform is ready for processing by Whisper
print(f"Waveform Shape after resampling: {waveform.shape}")

Resampling from 44100 Hz to 16kHz
Waveform Shape after resampling: torch.Size([2, 159680])


In [ ]:
import torch

# Check if CUDA (GPU) is available
print("CUDA available:", torch.cuda.is_available())
print("Using device:", "GPU" if torch.cuda.is_available() else "CPU")

CUDA available: True
Using device: GPU


In [ ]:
import torchaudio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import os
from tqdm import tqdm

# Initialize the fine-tuned Whisper model and processor
fine_tuned_model_name = "MAdel121/whisper-small-egyptian-arabic"  # Fine-tuned model
processor = WhisperProcessor.from_pretrained(fine_tuned_model_name)
model = WhisperForConditionalGeneration.from_pretrained(fine_tuned_model_name)

# Ensure using GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Function to load, resample, and transcribe audio using fine-tuned model
def transcribe_finetuned_audio(audio_path):
    # Load audio with torchaudio
    waveform, sample_rate = torchaudio.load(audio_path)

    # Print the shape of the waveform before any processing
    print(f"Original waveform shape: {waveform.shape}")

    # If the sample rate is not 16kHz, resample to 16kHz
    if sample_rate != 16000:
        print(f"Resampling from {sample_rate} Hz to 16kHz")
        waveform = torchaudio.transforms.Resample(sample_rate, 16000)(waveform)
        sample_rate = 16000

    # Convert multi-channel audio to mono if necessary
    # Whisper models expect single-channel audio.
    if waveform.shape[0] > 1: # Check if it's multi-channel (e.g., [2, N])
        print(f"Converting multi-channel audio to mono by averaging channels. Original shape: {waveform.shape}")
        waveform = waveform.mean(dim=0) # Average across channels to get a 1D tensor (N,)

    # Ensure the waveform is a 1D tensor (mono audio samples).
    # The processor expects a 1D tensor for a single audio input.
    if len(waveform.shape) > 1: # If it's (1, N) (e.g., mono with a channel dim), squeeze to (N,)
        waveform = waveform.squeeze()

    print(f"Final waveform shape before processor: {waveform.shape}")

    # Process the audio input with Whisper processor
    # The processor will handle adding a batch dimension if needed.
    audio_input = processor(waveform, return_tensors="pt", sampling_rate=16000).input_features.to(device)

    # Generate transcription
    predicted_ids = model.generate(audio_input)
    transcription = processor.decode(predicted_ids[0], skip_special_tokens=True)

    return transcription

# Test on a single audio file to verify the transcription works

# =====
# test_audio_path = "/content/NileTTS-dataset/wavs/sales_34_00019.wav"  # Replace with your test file
test_audio_path = str(DATASET_DIR / "wavs" / "sales_34_00019.wav")
# =====

print("Transcribing:", test_audio_path)

# Transcribe and print the result
transcription = transcribe_finetuned_audio(test_audio_path)
print("Transcription:", transcription)


preprocessor_config.json:   0%|          | 0.00/371 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json: 0.00B [00:00, ?B/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Transcribing: /content/NileTTS-dataset/wavs/sales_34_00019.wav
Original waveform shape: torch.Size([2, 440118])
Resampling from 44100 Hz to 16kHz
Converting multi-channel audio to mono by averaging channels. Original shape: torch.Size([2, 159680])
Final waveform shape before processor: torch.Size([159680])


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

Transcription:  فالشخص الذي هو يعني حفظ مش فاهم ده اللي قافل على نفسه وبعمل حاجة واحدة بس ده ممكن يبقى مشكلة بعدين. الأهم دلوقتي الشخص اللي عنده فضول.


In [ ]:
# Apply transcription to eval dataset with fine-tuned model
tqdm.pandas()
eval_df['asr_transcription_finetuned'] = eval_df['audio_path'].progress_apply(lambda x: transcribe_finetuned_audio(x))

# Save the results

# =====
output_file = ASR_DIR / "asr_eval_finetuned_transcriptions.csv"

# eval_df.to_csv("/content/asr_eval_finetuned_transcriptions.csv", index=False)
eval_df.to_csv(output_file, index=False)

# print("Transcriptions saved to /content/asr_eval_finetuned_transcriptions.csv")
print("Saved:", output_file)
# =====


  0%|          | 0/950 [00:00<?, ?it/s]

Original waveform shape: torch.Size([2, 440118])
Resampling from 44100 Hz to 16kHz
Converting multi-channel audio to mono by averaging channels. Original shape: torch.Size([2, 159680])
Final waveform shape before processor: torch.Size([159680])


  0%|          | 2/950 [00:01<10:36,  1.49it/s]

Original waveform shape: torch.Size([2, 667673])
Resampling from 44100 Hz to 16kHz
Converting multi-channel audio to mono by averaging channels. Original shape: torch.Size([2, 242240])
Final waveform shape before processor: torch.Size([242240])


  0%|          | 3/950 [00:03<18:03,  1.14s/it]

...

Original waveform shape: torch.Size([2, 459522])
Resampling from 44100 Hz to 16kHz
Converting multi-channel audio to mono by averaging channels. Original shape: torch.Size([2, 166720])
Final waveform shape before processor: torch.Size([166720])


100%|██████████| 950/950 [26:25<00:00,  1.65s/it]

Original waveform shape: torch.Size([2, 622692])
Resampling from 44100 Hz to 16kHz
Converting multi-channel audio to mono by averaging channels. Original shape: torch.Size([2, 225920])
Final waveform shape before processor: torch.Size([225920])


100%|██████████| 950/950 [26:27<00:00,  1.67s/it]

Transcriptions saved to /content/asr_eval_finetuned_transcriptions.csv


In [ ]:
# Normalize both reference and ASR transcription for Fine-tuned model
#eval_df['ref_norm'] = eval_df['text'].apply(normalize_arabic)
eval_df['asr_norm_finetuned'] = eval_df['asr_transcription_finetuned'].apply(normalize_arabic)

In [ ]:
# Compute WER and CER for Fine-tuned model
eval_df['wer_finetuned'] = eval_df.apply(lambda row: wer(row['ref_norm'], row['asr_norm_finetuned']), axis=1)
eval_df['cer_finetuned'] = eval_df.apply(lambda row: cer(row['ref_norm'], row['asr_norm_finetuned']), axis=1)

# Print average WER and CER for Fine-tuned model
print(f"Fine-Tuned Model - WER: {eval_df['wer_finetuned'].mean()}")
print(f"Fine-Tuned Model - CER: {eval_df['cer_finetuned'].mean()}")

Fine-Tuned Model - WER: 0.37214257883552326
Fine-Tuned Model - CER: 0.15727691188171475


In [ ]:
# Compare results
results = {
    "Base Model WER": eval_df['wer_base'].mean(),
    "Base Model CER": eval_df['cer_base'].mean(),

    "Fine-Tuned WER": eval_df['wer_finetuned'].mean(),
    "Fine-Tuned CER": eval_df['cer_finetuned'].mean(),
}

for model, score in results.items():
    print(f"{model}: {score}")

Base Model WER: 0.5363087785012708
Base Model CER: 0.17257424279652153
Fine-Tuned WER: 0.37214257883552326
Fine-Tuned CER: 0.15727691188171475
